# 93 — Hybrid: Liquidation + SOPR Confluence

**Hypothesis**: Requiring BOTH a liquidation cascade AND STH-SOPR capitulation filters out noise and produces higher-conviction entries.

## Strategy
- **Resolution**: H4 (balance between noise and signal)
- **Entry**: Long liquidation spike (Z > threshold) AND STH-SOPR < threshold (both on same bar or within N bars)
- **Exit**: STH-SOPR > 1.0 OR take profit OR stop loss
- **Risk**: High — fewer signals but each is higher conviction
- **Period**: 2020-02-02 → present

This is the BTD logic compressed to intraday — combining notebooks 90 + 91.

In [ ]:
import pandas as pd
import numpy as np
import vectorbt as vbt
from pathlib import Path
from itertools import product
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('.').resolve().parent
BL_H4 = PROJECT_ROOT / 'data' / 'bl' / 'h4'
BL_HOURLY = PROJECT_ROOT / 'data' / 'bl' / 'hourly'
GN_HOURLY = PROJECT_ROOT / 'data' / 'glassnode' / 'hourly'

BACKTEST_START = '2020-02-02'
print(f'Project root: {PROJECT_ROOT}')

## 1. Data Loading

In [ ]:
def load_parquet(path: Path) -> pd.Series:
    if not path.exists():
        print(f'WARNING: {path} not found')
        return pd.Series(dtype=float)
    df = pd.read_parquet(path)
    time_col = next((c for c in ['time', 'date', 'timestamp'] if c in df.columns), None)
    if time_col is None and isinstance(df.index, pd.DatetimeIndex):
        df = df.reset_index()
        time_col = df.columns[0]
    val_col = 'value' if 'value' in df.columns else next((c for c in df.columns if c not in ['time', 'date', 'timestamp']), None)
    s = df.set_index(time_col)[val_col].sort_index()
    s.index = pd.to_datetime(s.index)
    if s.index.tz is not None:
        s.index = s.index.tz_localize(None)
    return s.astype(float)


# Load H4 on-chain data
price_h4 = load_parquet(BL_H4 / 'price.parquet').loc[BACKTEST_START:]
sopr_sth_h4 = load_parquet(BL_H4 / 'sopr_sth.parquet')
mvrv_sth_h4 = load_parquet(BL_H4 / 'mvrv_sth.parquet')
mvrv_h4 = load_parquet(BL_H4 / 'mvrv.parquet')

# Load hourly derivatives
liq_long = load_parquet(GN_HOURLY / 'liquidations_long.parquet')
liq_short = load_parquet(GN_HOURLY / 'liquidations_short.parquet')
funding = load_parquet(GN_HOURLY / 'funding_rate.parquet')

# Build H4 DataFrame
idx = price_h4.index
df = pd.DataFrame(index=idx)
df['price'] = price_h4
df['sopr_sth'] = sopr_sth_h4.reindex(idx, method='ffill')
df['mvrv_sth'] = mvrv_sth_h4.reindex(idx, method='ffill')
df['mvrv'] = mvrv_h4.reindex(idx, method='ffill')
df['liq_long'] = liq_long.reindex(idx, method='ffill')
df['liq_short'] = liq_short.reindex(idx, method='ffill')
df['funding'] = funding.reindex(idx, method='ffill')
df = df.dropna(subset=['price'])

print(f'H4 bars: {len(df):,}')
print(f'Date range: {df.index[0]} → {df.index[-1]}')
print(f'Nulls: sopr_sth={df["sopr_sth"].isna().sum()}, liq_long={df["liq_long"].isna().sum()}')

## 2. Individual Signal Components

In [ ]:
# Liquidation spike detection (Z-score)
LOOKBACK = 42  # 7 days in H4 bars (42 = 168h / 4h)
df['liq_long_ma'] = df['liq_long'].rolling(LOOKBACK, min_periods=6).mean()
df['liq_long_std'] = df['liq_long'].rolling(LOOKBACK, min_periods=6).std()
df['liq_long_z'] = (df['liq_long'] - df['liq_long_ma']) / df['liq_long_std'].replace(0, np.nan)

# Forward returns
for bars in [1, 3, 6, 12, 18, 36]:  # 4h to 6d
    df[f'fwd_{bars}b'] = df['price'].pct_change(bars).shift(-bars) * 100

# Compare: individual signals vs confluence
print('Forward returns (H4 bars → hours) by signal type:')
print('=' * 100)
bar_labels = [1, 3, 6, 12, 18, 36]
hour_labels = [4, 12, 24, 48, 72, 144]

print(f'{"Signal":>35s}', end='')
for h in hour_labels:
    print(f'  |  {h:>5}h', end='')
print(f'  |  {"N":>5}')
print('-' * 100)

signals = {
    'Liq Z > 2.0':               df['liq_long_z'] > 2.0,
    'STH-SOPR < 0.95':           df['sopr_sth'] < 0.95,
    'STH-SOPR < 0.97':           df['sopr_sth'] < 0.97,
    'Funding < 0':               df['funding'] < 0,
    'Liq Z>2 + SOPR<0.97':       (df['liq_long_z'] > 2.0) & (df['sopr_sth'] < 0.97),
    'Liq Z>2 + SOPR<0.95':       (df['liq_long_z'] > 2.0) & (df['sopr_sth'] < 0.95),
    'Liq Z>1.5 + SOPR<0.97':     (df['liq_long_z'] > 1.5) & (df['sopr_sth'] < 0.97),
    'All 3: Z>2+SOPR<0.97+F<0':  (df['liq_long_z'] > 2.0) & (df['sopr_sth'] < 0.97) & (df['funding'] < 0),
    'All bars (baseline)':        pd.Series(True, index=df.index),
}

for label, mask in signals.items():
    n = mask.sum()
    print(f'{label:>35s}', end='')
    for bars in bar_labels:
        col = f'fwd_{bars}b'
        ret = df.loc[mask, col].mean() if n > 0 else 0
        print(f'  |  {ret:>+5.2f}%', end='')
    print(f'  |  {n:>5}')

In [ ]:
# Visualize confluence events on price chart
confluence = (df['liq_long_z'] > 2.0) & (df['sopr_sth'] < 0.97)

fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)

# Price with entry markers
axes[0].plot(df.index, df['price'], color='#3b82f6', linewidth=0.5)
entries = df[confluence]
axes[0].scatter(entries.index, entries['price'], color='lime', s=30, zorder=5, marker='^',
                label=f'Confluence entries ({len(entries)})')
axes[0].set_ylabel('Price ($)')
axes[0].set_title('Hybrid Liquidation + SOPR Confluence (H4)')
axes[0].set_yscale('log')
axes[0].legend()

# Liq Z-score
axes[1].plot(df.index, df['liq_long_z'], color='#ef4444', linewidth=0.3)
axes[1].axhline(2.0, color='red', linestyle='--')
axes[1].set_ylabel('Liq Long Z-score')

# STH-SOPR
axes[2].plot(df.index, df['sopr_sth'], color='#8b5cf6', linewidth=0.3)
axes[2].axhline(0.97, color='red', linestyle='--')
axes[2].axhline(1.0, color='black', linestyle='-', alpha=0.5)
axes[2].set_ylabel('STH-SOPR')
axes[2].set_ylim(0.85, 1.15)

# Funding
axes[3].plot(df.index, df['funding'], color='#22c55e', linewidth=0.3)
axes[3].axhline(0, color='black', linestyle='-', alpha=0.5)
axes[3].set_ylabel('Funding Rate')

plt.tight_layout()
plt.show()

## 3. Lookahead Window — Confluence Within N Bars

Liquidation and SOPR may not spike on the exact same bar. Test allowing a window of N bars for the conditions to co-occur.

In [ ]:
# Test: require both conditions within N bars of each other
# Use rolling max of liq Z and rolling min of SOPR within N bars

print('Confluence frequency with rolling window:')
print('=' * 80)

for window in [1, 3, 6, 12]:  # 4h, 12h, 24h, 48h
    liq_rolling_max = df['liq_long_z'].rolling(window, min_periods=1).max()
    sopr_rolling_min = df['sopr_sth'].rolling(window, min_periods=1).min()
    
    for liq_z in [1.5, 2.0, 2.5]:
        for sopr_t in [0.95, 0.97]:
            mask = (liq_rolling_max > liq_z) & (sopr_rolling_min < sopr_t)
            n = mask.sum()
            # Mean 48h forward return
            fwd = df.loc[mask, 'fwd_12b'].mean() if n > 0 else 0  # 12 h4 bars = 48h
            print(f'  Win={window*4:>3}h  Z>{liq_z}  SOPR<{sopr_t}:  {n:>4} bars  |  48h fwd: {fwd:>+.2f}%')

## 4. VectorBT Backtest — Grid Search

In [ ]:
def run_confluence_backtest(df, liq_z_thresh, sopr_thresh, window_bars,
                            exit_sopr=1.0, sl_pct=None, tp_pct=None,
                            add_funding=False):
    """Backtest hybrid liquidation + SOPR confluence strategy.
    
    Entry: rolling_max(liq_z, window) > liq_z_thresh AND 
           rolling_min(sopr_sth, window) < sopr_thresh
           [Optional: AND funding < 0]
    Exit: sopr_sth > exit_sopr OR tp/sl
    """
    price = df['price'].copy()
    
    liq_rolling = df['liq_long_z'].rolling(window_bars, min_periods=1).max()
    sopr_rolling = df['sopr_sth'].rolling(window_bars, min_periods=1).min()
    
    entries = (liq_rolling > liq_z_thresh) & (sopr_rolling < sopr_thresh)
    if add_funding:
        entries = entries & (df['funding'] < 0)
    
    exits = df['sopr_sth'] > exit_sopr
    
    if entries.sum() == 0:
        return None
    
    kwargs = dict(
        close=price,
        entries=entries,
        exits=exits,
        fees=0.001,
        slippage=0.001,
        init_cash=10_000,
        freq='4h',
    )
    
    if sl_pct is not None:
        kwargs['sl_stop'] = sl_pct / 100
    if tp_pct is not None:
        kwargs['tp_stop'] = tp_pct / 100
    
    pf = vbt.Portfolio.from_signals(**kwargs)
    n_trades = pf.trades.count()
    if n_trades == 0:
        return None
    
    return {
        'liq_z': liq_z_thresh,
        'sopr': sopr_thresh,
        'window': window_bars,
        'exit_sopr': exit_sopr,
        'sl_pct': sl_pct,
        'tp_pct': tp_pct,
        'funding_filter': add_funding,
        'total_return': pf.total_return() * 100,
        'sharpe': pf.sharpe_ratio(),
        'max_dd': pf.max_drawdown() * 100,
        'n_trades': n_trades,
        'win_rate': pf.trades.win_rate() * 100,
        'avg_trade': pf.trades.returns.mean() * 100,
        'portfolio': pf,
    }


# Grid search
liq_z_levels = [1.5, 2.0, 2.5]
sopr_levels = [0.93, 0.95, 0.97]
windows = [1, 3, 6]  # 4h, 12h, 24h
exit_soprs = [1.0, 1.02]
sl_levels = [None, 3, 5]
tp_levels = [None, 3, 5, 10]
funding_filters = [False, True]

grid_results = []
total = len(liq_z_levels) * len(sopr_levels) * len(windows) * len(exit_soprs) * len(sl_levels) * len(tp_levels) * len(funding_filters)
print(f'Running {total} parameter combinations...')

for lz, sp, w, ex, sl, tp, ff in product(liq_z_levels, sopr_levels, windows, exit_soprs, sl_levels, tp_levels, funding_filters):
    r = run_confluence_backtest(df, lz, sp, w, ex, sl, tp, ff)
    if r is not None:
        grid_results.append(r)

print(f'Completed: {len(grid_results)} valid results out of {total} combos')

In [ ]:
# Results table — top 20 by Sharpe
grid_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'portfolio'} for r in grid_results])
grid_df = grid_df.sort_values('sharpe', ascending=False)

print('Top 20 configurations by Sharpe ratio:')
print('=' * 150)
print(f'{"LiqZ":>5s}  {"SOPR":>5s}  {"Win":>4s}  {"ExSO":>5s}  {"SL%":>4s}  {"TP%":>4s}  {"Fund":>5s}  |  {"Return":>9s}  {"Sharpe":>7s}  {"MaxDD":>8s}  {"Trades":>7s}  {"WinRate":>8s}  {"AvgTrade":>9s}')
print('-' * 150)

for _, row in grid_df.head(20).iterrows():
    sl_str = f'{row["sl_pct"]:.0f}' if row['sl_pct'] is not None and not (isinstance(row['sl_pct'], float) and np.isnan(row['sl_pct'])) else '-'
    tp_str = f'{row["tp_pct"]:.0f}' if row['tp_pct'] is not None and not (isinstance(row['tp_pct'], float) and np.isnan(row['tp_pct'])) else '-'
    ff_str = 'Yes' if row['funding_filter'] else 'No'
    print(f'{row["liq_z"]:>5.1f}  {row["sopr"]:>5.2f}  {row["window"]*4:>3.0f}h  {row["exit_sopr"]:>5.2f}  {sl_str:>4s}  {tp_str:>4s}  {ff_str:>5s}  |  '
          f'{row["total_return"]:>+8.1f}%  {row["sharpe"]:>7.2f}  {row["max_dd"]:>7.1f}%  '
          f'{row["n_trades"]:>7.0f}  {row["win_rate"]:>7.1f}%  {row["avg_trade"]:>+8.2f}%')

## 5. Best Configuration Deep Dive

In [ ]:
best = max(grid_results, key=lambda r: r['sharpe'] if r['sharpe'] == r['sharpe'] else -999)
pf_best = best['portfolio']

print(f'Best configuration:')
print(f'  Liq Z threshold: > {best["liq_z"]}')
print(f'  SOPR threshold:  < {best["sopr"]}')
print(f'  Window:          {best["window"]*4}h ({best["window"]} bars)')
print(f'  Exit SOPR:       > {best["exit_sopr"]}')
print(f'  Stop loss:       {best["sl_pct"]}%')
print(f'  Take profit:     {best["tp_pct"]}%')
print(f'  Funding filter:  {best["funding_filter"]}')
print(f'\nPerformance:')
print(f'  Total return: {best["total_return"]:+.1f}%')
print(f'  Sharpe ratio: {best["sharpe"]:.2f}')
print(f'  Max drawdown: {best["max_dd"]:.1f}%')
print(f'  Trades:       {best["n_trades"]}')
print(f'  Win rate:     {best["win_rate"]:.1f}%')
print(f'  Avg trade:    {best["avg_trade"]:+.2f}%')

trades = pf_best.trades.records_readable
if len(trades) > 0:
    print(f'\nTrade log (all):')
    cols = [c for c in ['Entry Timestamp', 'Exit Timestamp', 'PnL', 'Return', 'Status'] if c in trades.columns]
    print(trades[cols].to_string())

## 6. Equity Curve

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8), gridspec_kw={'height_ratios': [3, 1]})

equity = pf_best.value()
bh = 10_000 * df['price'] / df['price'].iloc[0]

axes[0].plot(equity.index, equity.values, color='#f59e0b',
             label=f'Confluence ({best["total_return"]:+.0f}%)')
axes[0].plot(bh.index, bh.values, color='#ef4444', linestyle='--', alpha=0.5,
             label=f'Buy & Hold ({(bh.iloc[-1]/10000-1)*100:+.0f}%)')
axes[0].set_ylabel('Portfolio Value ($)')
axes[0].set_title(f'Hybrid Liq+SOPR Confluence (Z>{best["liq_z"]}, SOPR<{best["sopr"]}, win={best["window"]*4}h)')
axes[0].legend()
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3)

dd = pf_best.drawdown()
axes[1].fill_between(dd.index, dd.values * 100, 0, color='#f59e0b', alpha=0.3)
axes[1].set_ylabel('Drawdown (%)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Compare: Individual Signals vs Confluence

In [ ]:
# Run backtests for individual signals with same exit for comparison
comparison = {}

# Liq only
r_liq = run_confluence_backtest(df, best['liq_z'], 999, best['window'],  # sopr=999 → always true
                                 best['exit_sopr'], best['sl_pct'], best['tp_pct'], best['funding_filter'])

# SOPR only — use a very low liq threshold so it always passes
r_sopr = run_confluence_backtest(df, -999, best['sopr'], best['window'],  # liq=-999 → always true
                                  best['exit_sopr'], best['sl_pct'], best['tp_pct'], best['funding_filter'])

# Confluence (best)
r_conf = best

print('Individual vs Confluence Comparison:')
print('=' * 100)
print(f'{"Signal":>25s}  |  {"Return":>9s}  {"Sharpe":>7s}  {"MaxDD":>8s}  {"Trades":>7s}  {"WinRate":>8s}  {"AvgTrade":>9s}')
print('-' * 100)

for label, r in [('Liq only', r_liq), ('SOPR only', r_sopr), ('Confluence', r_conf)]:
    if r is not None:
        print(f'{label:>25s}  |  {r["total_return"]:>+8.1f}%  {r["sharpe"]:>7.2f}  {r["max_dd"]:>7.1f}%  '
              f'{r["n_trades"]:>7}  {r["win_rate"]:>7.1f}%  {r["avg_trade"]:>+8.2f}%')
    else:
        print(f'{label:>25s}  |  No trades')

## 8. Key Findings

In [ ]:
print('=' * 70)
print('HYBRID LIQ + SOPR CONFLUENCE — KEY FINDINGS')
print('=' * 70)

print(f'\n1. CONFLUENCE FREQUENCY:')
for lz in [1.5, 2.0, 2.5]:
    for sp in [0.95, 0.97]:
        n = ((df['liq_long_z'] > lz) & (df['sopr_sth'] < sp)).sum()
        print(f'   Z>{lz} + SOPR<{sp}: {n} bars ({n/len(df)*100:.2f}%)')

print(f'\n2. BEST CONFIGURATION:')
print(f'   Z>{best["liq_z"]} + SOPR<{best["sopr"]} | Window: {best["window"]*4}h | Exit: SOPR>{best["exit_sopr"]}')
print(f'   SL: {best["sl_pct"]}% | TP: {best["tp_pct"]}% | Funding filter: {best["funding_filter"]}')
print(f'   Return: {best["total_return"]:+.1f}% | Sharpe: {best["sharpe"]:.2f} | MaxDD: {best["max_dd"]:.1f}%')
print(f'   Trades: {best["n_trades"]} | Win rate: {best["win_rate"]:.1f}% | Avg trade: {best["avg_trade"]:+.2f}%')

print(f'\n3. INDIVIDUAL vs CONFLUENCE:')
if r_liq:
    print(f'   Liq only:    Return {r_liq["total_return"]:+.1f}% | Sharpe {r_liq["sharpe"]:.2f} | {r_liq["n_trades"]} trades')
if r_sopr:
    print(f'   SOPR only:   Return {r_sopr["total_return"]:+.1f}% | Sharpe {r_sopr["sharpe"]:.2f} | {r_sopr["n_trades"]} trades')
print(f'   Confluence:  Return {best["total_return"]:+.1f}% | Sharpe {best["sharpe"]:.2f} | {best["n_trades"]} trades')

# Does confluence improve Sharpe?
conf_sharpe = best['sharpe']
liq_sharpe = r_liq['sharpe'] if r_liq else 0
sopr_sharpe = r_sopr['sharpe'] if r_sopr else 0
best_individual = max(liq_sharpe, sopr_sharpe)

print(f'\n4. GRID SEARCH SUMMARY:')
profitable = [r for r in grid_results if r['total_return'] > 0]
sharpes = [r['sharpe'] for r in grid_results if r['sharpe'] == r['sharpe']]
print(f'   Total configs: {len(grid_results)}')
print(f'   Profitable: {len(profitable)} ({len(profitable)/len(grid_results)*100:.0f}%)')
if sharpes:
    print(f'   Sharpe range: {min(sharpes):.2f} to {max(sharpes):.2f}')

print(f'\n5. VERDICT:')
if conf_sharpe > best_individual and conf_sharpe > 0.5:
    print('   Confluence IMPROVES over individual signals.')
    print(f'   Sharpe uplift: {conf_sharpe:.2f} vs best individual {best_individual:.2f}')
    print('   The combination filters noise and produces higher-conviction entries.')
elif conf_sharpe > 0.5:
    print('   Confluence works but doesn\'t clearly beat individual signals.')
    print('   Additional complexity may not be justified.')
else:
    print('   Confluence does not produce a tradeable edge.')
    print('   The signals are too rare or the relationship is too noisy.')

print('\n' + '=' * 70)